
# Regression — Прогноз месячных расходов клиента

**Цель:** показать регрессию на табличных данных.  
**Данные:** синтетические признаки (`num_purchases`, `avg_bill`, `days_active`, `segment`) → `monthly_spend`.  
**Модель:** `LinearRegression`.  
**Метрики:** `MAE`, `MSE`, `R2`.

> Формат: всё самодостаточно, без загрузки внешних файлов.


In [ ]:
# Импорты
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)
print("Импорты готовы")


## 1. Генерация синтетических данных


In [ ]:
n = 300
num_purchases = np.random.poisson(lam=8, size=n).clip(0, 30)
avg_bill = np.random.normal(loc=1200, scale=400, size=n).clip(200, 4000)
days_active = np.random.randint(5, 31, size=n)
segment = np.random.choice(["new", "regular", "vip"], size=n, p=[0.3, 0.6, 0.1])

segment_effect = np.where(segment == "vip", 1.25, np.where(segment == "regular", 1.0, 0.85))

noise = np.random.normal(0, 5000, size=n)
monthly_spend = (num_purchases * avg_bill * 0.8 * segment_effect) + (days_active * 50) + noise
monthly_spend = np.clip(monthly_spend, 0, None)

df = pd.DataFrame({
    "num_purchases": num_purchases,
    "avg_bill": avg_bill,
    "days_active": days_active,
    "segment": segment,
    "monthly_spend": monthly_spend
})

df.head()


## 2. Быстрый EDA


In [ ]:
plt.figure(figsize=(6,4))
plt.hist(df["monthly_spend"], bins=30)
plt.title("Распределение monthly_spend")
plt.xlabel("monthly_spend")
plt.ylabel("count")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(df["num_purchases"], df["monthly_spend"])
plt.title("num_purchases vs monthly_spend")
plt.xlabel("num_purchases")
plt.ylabel("monthly_spend")
plt.show()


## 3. Подготовка признаков и Train/Test split


In [ ]:
df_enc = pd.get_dummies(df, columns=["segment"], drop_first=True)

X = df_enc.drop(columns=["monthly_spend"])
y = df_enc["monthly_spend"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape


## 4. Обучение модели и метрики


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae:.2f}\nMSE: {mse:.2f}\nR2: {r2:.3f}")

In [ ]:
plt.figure(figsize=(6,4))
plt.scatter(y_test, y_pred)
plt.title("Факт vs Прогноз (Test)")
plt.xlabel("Факт (y_test)")
plt.ylabel("Прогноз (y_pred)")
plt.show()


## 5. Вопросы для обсуждения
1) Что сильнее влияет на `monthly_spend`: количество покупок или средний чек?  
2) Почему для регрессии мы оцениваем `MAE/MSE/R2`, а не `accuracy`?  
3) Какие новые признаки добавили бы из практики (например, сезонность, канал, скидки)?  
4) Где регрессия применима в наших задачах (LTV, прогноз выручки, средний чек)?
